In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/imdb/annotated/checkpoints/post_cell_2.pickle

trying: ['plt']
me:  0
trying: ['np']
me:  0
trying: ['factor']
me:  3
trying: ['time']
me:  0
trying: ['filename']
me:  1
trying: ['orig_output']
me:  2
trying: ['pd']
me:  0
trying: ['m']


me:  3


Declaring variable plt
Declaring variable np
Declaring variable time
Declaring variable pd
Declaring variable filename
Declaring variable orig_output
Declaring variable factor
Declaring variable m


In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Trim whitespace on the movie titles
m.movie_title = m.movie_title.str.strip()

# Convert all numeric columns in one go, using int64 for duration to match the original pd.to_numeric behavior
m = m.astype({
    'duration':   'int64',
    'budget':     'float64',
    'gross':      'float64',
    'imdb_score': 'float64'
})

# Re-index and drop all unwanted columns in a single call each
tmp_index = 'movie_title'
m.set_index(tmp_index, inplace=True)
cols_to_drop = [
    'color', 'director_facebook_likes', 'actor_3_facebook_likes',
    'actor_2_name', 'actor_1_facebook_likes', 'actor_1_name',
    'cast_total_facebook_likes', 'movie_imdb_link', 'language',
    'actor_2_facebook_likes', 'aspect_ratio', 'actor_3_name',
    'facenumber_in_poster', 'plot_keywords', 'country'
]
m.drop(columns=cols_to_drop, inplace=True)


CPU times: user 4.27 ms, sys: 16.1 ms, total: 20.4 ms
Wall time: 20.4 ms


In [5]:
%Checkpoint /home/dias-benchmarks/new_notebooks/imdb/rewritten/o4_mini_high/checkpoints/post_cell_3_try_1.pickle

migration speed (bps): 757431258.4175454
---------------------------
variables to migrate:
orig_output 16
np 72
filename 108
tmp_index 60
factor 28
m 48
cols_to_drop 184
plt 72
pd 72
time 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/new_notebooks/imdb/rewritten/o4_mini_high/checkpoints/post_cell_3_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['m']
Intermediate variables ['filename']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['m']
Intermediate variables ['factor']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['m']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['cols_to_drop', 'm', 'tmp_index']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:
with open(
    "/home/dias-benchmarks/new_notebooks/imdb/opt_cell_exec_info_3_try_1.pkl", "wb"
) as f:
    pickle.dump(opt_cell_exec_info[3], f)

In [ ]:
opt_output = Out.get(4)

In [ ]:
m_opt = m
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/imdb/annotated/checkpoints/post_cell_3.pickle
assert compare_df(m_opt, m)

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output